# Make data for BioGeoBEARS

## Loading packages

In [13]:
library("tidygraph")
library("igraph")
library("ggraph")
library("tidyverse")
library("ape")


Attachement du package : ‘ape’


L'objet suivant est masqué depuis ‘package:dplyr’:

    where


Les objets suivants sont masqués depuis ‘package:igraph’:

    degree, edges, mst, ring




## Loading data

In [2]:
table <- read.table("../Data_orectolobiformes/Time_periods.tsv", sep ="\t", header = TRUE)

In [ ]:
table_biogeo <- read.table("../Data_orectolobiformes/Biogeography extant.tsv", sep ="\t", header = TRUE)

In [14]:
phy <- read.tree("../Data_orectolobiformes/tree_orecto.tree")

In [17]:
table_biogeo[gsub(" ", "_", table_biogeo$Species) %in% phy$tip.label, ]

,Family,Genus,Species,East.Pacific,West.Atlantic,East.Atlantic,Tethys,West.Indo.Pacific,Central.Indo.Pacific,Australasia
,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
1,Brachaeluridae,Brachaelurus,Brachaelurus colcloughi,0,0,0,0,0,0,1
2,Brachaeluridae,Brachaelurus,Brachaelurus waddi,0,0,0,0,0,0,1
3,Hemiscyliidae,Chiloscyllium,Chiloscyllium arabicum,0,0,0,0,1,0,0
4,Hemiscyliidae,Chiloscyllium,Chiloscyllium burmensis,0,0,0,0,1,0,0
5,Hemiscyliidae,Chiloscyllium,Chiloscyllium griseum,0,0,0,0,1,1,0
6,Hemiscyliidae,Chiloscyllium,Chiloscyllium hasseltii,0,0,0,0,0,1,0
7,Hemiscyliidae,Chiloscyllium,Chiloscyllium indicum,0,0,0,0,1,1,0
8,Hemiscyliidae,Chiloscyllium,Chiloscyllium plagiosum,0,0,0,0,1,1,0
9,Hemiscyliidae,Chiloscyllium,Chiloscyllium punctatum,0,0,0,0,1,1,1


In [ ]:
table_biogeo

In [12]:
cbind(gsub(as.matrix((apply(X = table_biogeo[,c(4:10)], MARGIN = 1, FUN = paste, collapse = "")))

0000001
0000001
0000100
0000100
0000110
0000010
0000110
0000110
0000111
0000010
0000010


## Prepare biogeographic data for extant taxa

In [7]:
table_biogeo <- read.table("../Data_orectolobiformes/Biogeography extant.tsv", sep ="\t", header = TRUE)


## Creating function to clean and save data

In [3]:
make_table_BioGeoBEARS <- function(time_periods_table, prefix){
    temp_length <- (ncol(time_periods_table) - 2)
    table_adjacency <- c()
    table_dispersal <- c()
    temp_mat_adj <- matrix(0, temp_length, temp_length)
    for(i in 1:nrow(time_periods_table)){
        temp_mat_adj <- matrix(0, temp_length, temp_length)
        temp_period_table <- time_periods_table[i, c(3:ncol(time_periods_table))]
        from <- c()
        to <- c()
        
        for(j in 1:temp_length){
            if(temp_period_table[j] != "0"){
                temp_mat_adj[j,eval(parse(text = temp_period_table[j]))] <- 1
                from <- c(from, rep(j,length(eval(parse(text = temp_period_table[j])))))
                to <- c(to, eval(parse(text = temp_period_table[j])))
            }
        }
        
        nodes <- tibble(id = 1:temp_length)
        edges <- tibble(from = from, to = to)
        
        temp_data_from <- edges[edges[,1] != edges[,2],]

        temp_data_to_1 <- edges[edges[,1] != edges[,2],]

        temp_data_to_2 <- edges[edges[,1] != edges[,2],]

        colnames(temp_data_to_1) <- c("to", "to_2")

        colnames(temp_data_to_2) <- c("to_2", "to_3")

        temp_data_03 <- merge(temp_data_from, temp_data_to_1, by = "to")
        temp_data_03 <- temp_data_03[temp_data_03[,2] != temp_data_03[,3],]

        temp_data_04 <- merge(temp_data_03, temp_data_to_1, by = "to_2")
        temp_data_04 <- temp_data_04[temp_data_04[,3] != temp_data_04[,4],]

        temp_mat_dispersal <- matrix(0.001, 7, 7)
        
        temp_data_direct <- as.matrix(edges)
        
        for(k in 1:nrow(temp_data_direct)){
            if(temp_data_direct[k,1] == temp_data_direct[k,2]){
                temp_mat_dispersal[temp_data_direct[k,1], temp_data_direct[k,2]] <- 1 
            }
            if(temp_data_direct[k,1] != temp_data_direct[k,2]){
                temp_mat_dispersal[temp_data_direct[k,1], temp_data_direct[k,2]] <- 0.5
            }
        
        }

        for(k in 1:nrow(temp_data_03)){
            if(temp_mat_dispersal[temp_data_03[k,2], temp_data_03[k,3]] == 0.0000001){
                temp_mat_dispersal[temp_data_03[k,2], temp_data_03[k,3]] <- 0.25
            }
        }

        for(k in 1:nrow(temp_data_04)){
            if(temp_mat_dispersal[temp_data_04[k,3], temp_data_04[k,4]] == 0.0000001){
                temp_mat_dispersal[temp_data_04[k,3], temp_data_04[k,4]] <- 0.125
            }
        }
        
        
        my_graph <- tbl_graph(nodes = nodes, edges = edges, directed = FALSE)
        
        connectivity_graph <-ggraph(my_graph, layout = "stress") +
            geom_edge_link() +
            geom_node_point(size = 10, colour = "antiquewhite") +
            geom_node_text(aes(label = id)) +
            theme_graph()
        
        ggsave(paste("Connection/connection_grap_", as.character(time_periods_table[i,2]), ".pdf",sep = ""))
        
        table_adjacency <- rbind(table_adjacency, LETTERS[1:7], temp_mat_adj, matrix(data=" ", ncol=7, nrow=1))
        
        table_dispersal <- rbind(table_dispersal, LETTERS[1:7], temp_mat_dispersal, matrix(data=" ", ncol=7, nrow=1))
    }
    table_adjacency <- rbind(table_adjacency, cbind("END", matrix(data = " ", ncol=6, nrow=1)))
    table_dispersal <- rbind(table_dispersal, cbind("END", matrix(data = " ", ncol=6, nrow=1)))
    write.table(time_periods_table[,2], paste(prefix, "_time_period.txt", sep = ""), sep ="\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
    write.table(table_adjacency, paste(prefix, "_area_matrix.txt", sep = ""), sep ="\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
    write.table(table_dispersal, paste(prefix, "_dispersal_matrix.txt", sep = ""), sep ="\t", row.names = FALSE, col.names = FALSE, quote = FALSE)
}

## Execute function

In [4]:
data <- make_table_BioGeoBEARS(table, "7_area")

Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
Saving 6.67 x 6.67 in image
